# StormEngine V8 — Stage 3 gradual unfreezing

Stage 3A freezes Encoder and trains Processor + Decoder. Stage 3B then jointly fine-tunes all modules with discriminative learning rates. Both seeds run sequentially on 2010–2015 training and 2016 validation; 2017 is never read. Re-running this cell resumes interrupted formal runs from `last.pt`.

In [ ]:
from pathlib import Path
import subprocess
import sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').is_file() else here.parent
assert (REPO / 'pyproject.toml').is_file(), REPO

PHASE = 'all'       # preflight | checks | stage3a | stage3b | all
SKIP_PILOT = False # set True only after the matching pilots have completed

command = [
    sys.executable, '-u', str(REPO / 'scripts' / 'run_v8_stage3.py'),
    '--phase', PHASE, '--device', 'cuda',
]
if SKIP_PILOT:
    command.append('--skip-pilot')
print('Running:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print('Stage 3 workflow completed.', flush=True)